# 01: Ball detection, one frame at a time

**Goal:** understand how the ball detector works by running each step on a single frame and *looking* at the result.

**Where this fits in the project:**

```
phone video  ->  [ BALL DETECTOR ]  ->  ball position in every frame  ->  (later) stroke analysis  ->  coaching voice
                   ^ we are here
```

The detector is not an AI model. It is 7 hand-written rules. A video is just ~60 photos per second, so if we can find the ball in one photo, we can find it in all of them by repeating the same steps in a loop. That loop is exactly what `scripts/ball_detect_baseline.py` does.

**How to use this notebook:** click a cell, press **Shift+Enter** to run it. Run them top to bottom. Steps 1-4 are done for you. **Steps 5-7 are yours.**

In [1]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLIP = ROOT / "data/raw/2026-10-17/IMG_1188.mov"
FRAME = 504      # a frame where the ball is clearly in flight against the wall
WIDTH = 960      # we shrink frames to this width: faster, and same as the script

def show(*imgs, titles=None, size=7):
    # Display one or more images side by side. Colour images are BGR (OpenCV's order), so convert for matplotlib.
    fig, axes = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size * 0.5))
    axes = np.atleast_1d(axes)
    for i, (ax, im) in enumerate(zip(axes, imgs)):
        if im.ndim == 3:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(im, cmap="gray")
        ax.set_title(titles[i] if titles else "")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Clip exists:", CLIP.exists())


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

## Step 1: Load two consecutive frames

To know what *moved*, we need the current frame and the one right before it (1/60th of a second earlier).

In [13]:
cap = cv2.VideoCapture(str(CLIP))
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME - 1)
_, prev = cap.read()
_, curr = cap.read()
cap.release()

h, w = curr.shape[:2]
prev = cv2.resize(prev, (WIDTH, int(h * WIDTH / w)))
curr = cv2.resize(curr, (WIDTH, int(h * WIDTH / w)))

print("Image shape (height, width, colour channels):", curr.shape)
show(prev, curr, titles=[f"frame {FRAME-1}", f"frame {FRAME}"])

NameError: name 'CLIP' is not defined

### Concept: an image is just a grid of numbers

`curr` is a NumPy array of shape `(height, width, 3)`. Each pixel has 3 numbers (Blue, Green, Red, each 0-255; OpenCV stores them in BGR order).
The ball in this frame is at roughly **x=605, y=165**. Here are its pixel values next to a wall pixel:

In [15]:
BALL_XY = (605, 165)   # (x, y) at 960px width, found by eye for frame 504
x, y = BALL_XY
print("ball pixel  (B, G, R):", curr[y, x])          # note: arrays are indexed [row, column] = [y, x]
print("wall pixel  (B, G, R):", curr[y, x - 60])

zoom = curr[y-40:y+40, x-60:x+60]
show(cv2.resize(zoom, None, fx=4, fy=4, interpolation=cv2.INTER_NEAREST), titles=["zoomed in on the ball (4x)"], size=5)

NameError: name 'curr' is not defined

## Step 2: Frame differencing (what moved?)

Convert both frames to **grayscale** (one brightness number per pixel, 0-255), then subtract them pixel by pixel.
Pixels that didn't change cancel out to ~0 (black). Pixels where something moved light up.

In [ ]:
prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)
diff = cv2.absdiff(prev_gray, curr_gray)      # |curr - prev| for every pixel

print("diff value at the ball:", diff[y, x], "  typical diff value:", np.median(diff))
show(curr_gray, diff, titles=["grayscale", "difference: bright = changed"])

## Step 3: Threshold (turn "how much changed" into yes/no)

Every pixel whose difference is above `MOTION_THRESHOLD` becomes white (255 = moved), everything else black (0).

**Try it:** change `MOTION_THRESHOLD` to 5, then 50, and rerun. Before you do, predict what will happen. Why might a value that's too low be a problem? Too high?

In [ ]:
MOTION_THRESHOLD = 18

_, motion = cv2.threshold(diff, MOTION_THRESHOLD, 255, cv2.THRESH_BINARY)

print(f"{(motion > 0).mean():.2%} of pixels counted as 'moved'")
print("ball pixel moved?", motion[y, x] == 255)
show(motion, titles=[f"motion mask (threshold={MOTION_THRESHOLD})"])

## Step 4: Colour thresholding (what's ball-coloured?)

RGB mixes colour and brightness together, so a ball in shade and a ball in sun have very different RGB values. **HSV** separates them:

| Channel | Meaning | OpenCV range |
|---|---|---|
| **H**ue | which colour (red, yellow, green...) | 0-179 |
| **S**aturation | how vivid vs grey | 0-255 |
| **V**alue | how bright | 0-255 |

Optic-yellow is roughly hue 25-45. `cv2.inRange` keeps pixels inside the box `BALL_LO <= pixel <= BALL_HI` on all three channels.

In [ ]:
hsv = cv2.cvtColor(curr, cv2.COLOR_BGR2HSV)
print("ball pixel (H, S, V):", hsv[y, x])
print("wall pixel (H, S, V):", hsv[y, x - 60])

BALL_LO = np.array([25, 80, 110])
BALL_HI = np.array([45, 255, 255])
color = cv2.inRange(hsv, BALL_LO, BALL_HI)

print(f"{(color > 0).mean():.2%} of pixels counted as 'ball-coloured'")
show(curr, color, titles=["frame", "colour mask: white = ball-coloured"])

**Look at the colour mask.** The ball is white, but what else is? This is the root of most false detections you saw in the video.

---

# Your turn

## Step 5: Combine the masks

A pixel should survive only if it **moved AND is ball-coloured**. Create `both` from `motion` and `color`.

Expected result: a mostly black image with a small white blob where the ball is (plus a little junk).

<details><summary>Hint 1</summary>Both masks are arrays of 0s and 255s with the same shape. You want a pixel-by-pixel logical AND.</details>
<details><summary>Hint 2</summary>OpenCV has a function for this in the <code>cv2.bitwise_...</code> family.</details>
<details><summary>Hint 3</summary><code>cv2.bitwise_and(a, b)</code>. Bonus: try <code>cv2.dilate(color, None, iterations=2)</code> first and see if the ball blob gets more complete. Why might that help with a blurry ball?</details>

In [17]:
both = None   # TODO: combine motion and color

if both is not None:
    print("ball pixel survived?", both[y, x] == 255)
    show(motion, color, both, titles=["moved", "ball-coloured", "moved AND ball-coloured"], size=5)

## Step 6: Contours (group pixels into blobs) and put them in a table

`cv2.findContours` traces the outline of each connected white region, so loose pixels become *objects*. The cell below finds them for you.

**Your job:** build a pandas DataFrame with one row per blob and these columns:

| column | how to get it |
|---|---|
| `x`, `y`, `r` | `(x, y), r = cv2.minEnclosingCircle(c)` (smallest circle around the blob) |
| `area` | `cv2.contourArea(c)` (pixels inside the outline) |
| `fill` | `area / (pi * r^2)`: how much of that circle the blob fills. 1.0 = perfect disc |

Then sort by `area` and look at it. Which row is the ball? (It should be near x=605, y=165.)

<details><summary>Hint 1</summary>Loop over <code>contours</code>, build a dict per blob, append it to a list, then <code>pd.DataFrame(list_of_dicts)</code>.</details>
<details><summary>Hint 2</summary>Watch out for <code>r = 0</code> on single-pixel blobs: add a tiny number like <code>1e-6</code> to the denominator.</details>

In [ ]:
contours = cv2.findContours(both, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0] if both is not None else []
print("number of blobs:", len(contours))

rows = []
# TODO: for each contour c, compute x, y, r, area, fill and append a dict to rows

blobs = pd.DataFrame(rows)
blobs

## Step 7: Filter and pick the ball

Now use the table to throw out blobs that can't be the ball, then pick one.

1. **Look at your table first.** What's the ball's `area` and `fill`? What do the junk blobs look like?
2. Choose `MIN_AREA`, `MAX_AREA` and `MIN_FILL` from what you see (this is you tuning the detector with data).
3. Filter `blobs` with pandas, pick the row with the highest `fill`, and draw a circle at its position.

<details><summary>Hint 1</summary>Boolean filtering: <code>blobs[(blobs.area >= MIN_AREA) & ...]</code></details>
<details><summary>Hint 2</summary><code>.sort_values("fill", ascending=False).iloc[0]</code> gives the best row. <code>cv2.circle(img, (int(x), int(y)), radius, (0, 0, 255), 2)</code> draws a red circle (BGR!). Draw on <code>curr.copy()</code> so you don't modify the original.</details>
<details><summary>Hint 3</summary>The script uses area between 0.002% and 0.3% of the frame, and fill >= 0.35. Are those good choices for this frame?</details>

In [ ]:
MIN_AREA = None   # TODO
MAX_AREA = None   # TODO
MIN_FILL = None   # TODO

# TODO: filter blobs, pick the best one, draw it on a copy of curr, and show it

---
## Experiments (once Steps 5-7 work)

Change `FRAME` in the first code cell and rerun everything (**Kernel > Restart & Run All**):

| FRAME | What's in it | Question |
|---|---|---|
| 504 | ball in flight against the wall | baseline: does your pipeline find it? |
| 1322 | ball right at the racket strings | does the racket interfere? |
| 67 | the detector picked something near the top-right fence here | what did it actually pick, and which filter should have stopped it? |

(`BALL_XY` is only correct for frame 504, so ignore the "ball pixel" prints on other frames.)

## Write down (2-3 sentences each, in the cell below)
1. Which step lets the most junk through?
2. Which step loses the real ball most easily?
3. If you could change **one** thing first, what would it be and why?

These answers decide our next experiment.

**My observations:**

1.
2.
3.